In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNBiLSTMClassifier(nn.Module):
    def __init__(
        self, 
        vocab_size, 
        embed_dim=128, 
        num_filters=64, 
        filter_sizes=(3, 4, 5), 
        hidden_dim=128, 
        num_classes=5, 
        dropout=0.3,
        padding_idx=0
    ):
        super().__init__()
        
        # 1. Embedding Layer
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        
        # 2. Parallel 1D CNNs for multi-scale n-gram detection
        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embed_dim, 
                out_channels=num_filters, 
                kernel_size=k, 
                padding=k // 2
            )
            for k in filter_sizes
        ])
        
        cnn_out_dim = num_filters * len(filter_sizes)
        
        # 3. Bidirectional LSTM to capture sequential context across extracted features
        self.lstm = nn.LSTM(
            input_size=cnn_out_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        # 4. Classification Head
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids):
        # input_ids shape: [batch_size, seq_len]
        embedded = self.embedding(input_ids)  # [batch_size, seq_len, embed_dim]
        
        # Conv1d expects shape: [batch_size, channels/embed_dim, seq_len]
        embedded = embedded.permute(0, 2, 1)
        
        # Apply 1D Convolutions + ReLU
        conv_outputs = [F.relu(conv(embedded)) for conv in self.convs]
        
        # Concatenate feature maps along channel dimension
        cnn_features = torch.cat(conv_outputs, dim=1)  # [batch_size, cnn_out_dim, seq_len]
        
        # Re-permute back for LSTM input: [batch_size, seq_len, cnn_out_dim]
        lstm_input = cnn_features.permute(0, 2, 1)
        
        # Pass through BiLSTM
        lstm_out, _ = self.lstm(lstm_input)  # [batch_size, seq_len, hidden_dim * 2]
        
        # Global Average Pooling over the reduced sequence dimension
        pooled = torch.mean(lstm_out, dim=1)  # [batch_size, hidden_dim * 2]
        
        # Final classification logits
        logits = self.fc(self.dropout(pooled))
        return logits